
# Chạy Lại `baseline_ctrl` λ = 0 Trên Colab — Bằng Code Hiện Tại

Chạy lại lần đối chứng λ = 0 với **đúng dòng lệnh của notebook vòng 1**, bằng bản code
hôm nay. Chạy thẳng 10 epoch, không có bước trung gian nào.

## Khác notebook vòng 1 đúng một chỗ: nơi ghi

Dòng lệnh huấn luyện chép từng cờ một từ `train_shift_consistency_vadclip_colab.ipynb`
mục 10. Cùng seed, cùng lr, cùng lô, cùng `--eval-steps 1280`, cùng `--num-workers 4`,
cùng luật chọn checkpoint.

**Không nên chạy chính notebook vòng 1** vì mục 10 của nó ghi thẳng đè lên
`model/model_baseline_ctrl.pth`, `model/epoch_checkpoints_baseline_ctrl/` và
`logs_shift_consistency/train_baseline_ctrl.log` trên Drive — nó không có bước "bỏ qua
nếu đã có". Chạy nó là xoá mất chứng cứ của chính lần chạy 88,13.

## Nơi ghi kết quả

**Mọi thứ ghi lên Drive đều nằm dưới `Result/shift_loss_09_09/`.** Hàm
`drive_write_path()` bọc mọi đường ghi và ném lỗi nếu có đường nào nằm trên Drive mà ra
ngoài thư mục đó. Không kết quả cũ nào bị đụng tới.

File trung gian nặng (checkpoint từng epoch, `model_cur`, checkpoint chọn-theo-AUC) ghi
vào `/content/scratch`, không lên Drive — mỗi cái 350 MB.

Tag là `ctrl_0909`, cố ý không trùng `baseline_ctrl`.

## ⚠️ Trước khi mở: upload lại `ucf_train_augment.py`

File này vừa được sửa dưới máy: dòng đặt `CUBLAS_WORKSPACE_CONFIG` giờ chỉ chạy khi lệnh
thật sự có `--deterministic true`. Trước đó nó chạy vô điều kiện, và đó là khác biệt duy
nhất không-được-chặn giữa code hôm nay và code đã chạy ra 88,13.

Upload `VadCLIP/src/ucf_train_augment.py` từ repo lên Drive
`Finetune VadCLIP/VadCLIP/src/`. Mục 3 dừng nếu Drive còn bản cũ.

## Chạy

Run all. Mục 1–5 là chuẩn bị, vài phút. **Mục 6 chạy thẳng 10 epoch.** Mục 7 đọc kết quả.



## 1. Mount Drive, Cấu Hình, Chốt Thư Mục Kết Quả

Ghi lại tên GPU mà Colab cấp cho phiên này. Log vòng 1 không ghi lại GPU, nên ta không
biết tháng 8 nó chạy trên card gì — biết card của lần này thì ít nhất lần sau còn so được.


In [ ]:

import hashlib
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Không chạy trong Colab, hoặc Drive đã được mount.')

print('Python:', sys.version.split()[0])
print('GPU   :', end=' ')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                '--format=csv,noheader'], check=False)

# --- Đường dẫn --------------------------------------------------------------------
PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR      = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR     = PROJECT_ROOT / 'VadCLIP' / 'list'
PAPER_MODEL  = PROJECT_ROOT / 'model_ucf.pth'

DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT

TRAIN_LIST = str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')
TEST_LIST  = str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv')
GT_ARGS = [
    '--gt-path',         str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path',   str(LIST_DIR / 'gt_label_ucf.npy'),
]

# ============ CẤU HÌNH — CHÉP TỪ baseline_ctrl CỦA NOTEBOOK VÒNG 1 ============
# Nguồn: code/train_shift_consistency_vadclip_colab.ipynb
#        mục 1  -> build_train_cmd(...)
#        mục 10 -> build_train_cmd(tag='baseline_ctrl', lambda_consistency='0', seed=234)
RUN_TAG        = 'ctrl_0909'   # tên riêng, không đụng 'baseline_ctrl' của vòng 1
SEED           = 234
LAMBDA         = '0'
MAX_EPOCH      = 10
LR             = '2e-5'
NUM_WORKERS    = 4             # PHẢI là 4: num_workers đổi thì thứ tự lô đổi theo
EVAL_STEPS     = 1280
USE_PRETRAINED = False
# =============================================================================

# ================= NƠI GHI KẾT QUẢ =================
OUT_DIR   = PROJECT_ROOT / 'Result' / 'shift_loss_09_09'
LOG_DIR   = OUT_DIR / 'logs'
MODEL_DIR = OUT_DIR / 'models'
SCRATCH   = Path('/content/scratch')      # file trung gian, KHÔNG lên Drive
BUILD_DIR = Path('/content/build')        # bản sao src, để không ghi __pycache__ lên Drive
# ===================================================

# Thư mục kết quả của các vòng trước. Notebook này không được chạm vào cái nào.
PROTECTED = {'Result', 'logs_shift_consistency', 'logs_shift_v2', 'logs_shift_v3',
             'ucf_checkpoint_diagnostics_baseline_ctrl', 'ucf_checkpoint_diagnostics_v0',
             'ucf_shift_sensitivity_baseline', 'ucf_shift_sensitivity_baseline_ctrl',
             'ucf_shift_sensitivity_v0'}
if OUT_DIR.name in PROTECTED:
    raise RuntimeError(f'OUT_DIR trùng tên một thư mục kết quả cũ: {OUT_DIR.name}')

for directory in (LOG_DIR, MODEL_DIR, SCRATCH):
    directory.mkdir(parents=True, exist_ok=True)


def drive_write_path(path):
    """Chốt chặn: mọi đường ghi lên Drive phải nằm dưới OUT_DIR.

    Kiểm bằng code, không bằng lời hứa. Gõ nhầm một đường dẫn thành thư mục kết quả cũ
    thì cell chết ngay tại đây, chứ không phải ghi đè xong mới biết.
    """
    path = Path(path)
    resolved = path.resolve() if path.exists() else (path.parent.resolve() / path.name)
    root, out = PROJECT_ROOT.resolve(), OUT_DIR.resolve()
    on_drive = root == resolved or root in resolved.parents
    inside_out = out == resolved or out in resolved.parents
    if on_drive and not inside_out:
        raise RuntimeError(f'Đường ghi nằm ngoài OUT_DIR, từ chối: {resolved}')
    return str(path)


print()
print('Kết quả về :', OUT_DIR)
print('File tạm   :', SCRATCH, '(không lên Drive)')
print('Lần chạy   :', RUN_TAG, '| seed', SEED, '| lambda', LAMBDA,
      '|', MAX_EPOCH, 'epoch | eval_steps', EVAL_STEPS, '| num_workers', NUM_WORKERS)


## 2. Dependencies

In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy matplotlib pandas


## 3. Kiểm File, Kiểm Phiên Bản Code, Dựng Cây Chạy

Ba việc, tất cả đều vài giây:

1. **Đủ file chưa.** Thiếu thì dừng ngay, không để phát hiện ở giờ thứ hai.
2. **`ucf_train_augment.py` trên Drive đã là bản đã sửa chưa.** Cell tìm hàm
   `_determinism_requested` trong file. Không thấy nghĩa là Drive còn bản đặt
   `CUBLAS_WORKSPACE_CONFIG` vô điều kiện — upload lại từ repo rồi chạy lại cell này.
3. **Copy `src/` sang `/content/build`.** Chạy thẳng trên Drive sẽ khiến Python ghi
   `__pycache__` vào Drive, tức ghi ngoài `OUT_DIR`. Chạy bản sao vừa tránh được điều đó
   vừa nhanh hơn nhiều vì import không phải đi qua Drive.


In [ ]:

required = [SRC_DIR / n for n in
            ['model.py', 'ucf_train_augment.py', 'ucf_option_augment.py',
             'ucf_test_description.py', 'ucf_train_class_prototype.py',
             'utils/dataset_augment.py', 'utils/tools.py', 'utils/layers.py',
             'utils/ucf_detectionMAP.py', 'clip/clip.py']]
required += [Path(TRAIN_LIST), Path(TEST_LIST),
             LIST_DIR / 'gt_ucf.npy', LIST_DIR / 'gt_segment_ucf.npy',
             LIST_DIR / 'gt_label_ucf.npy']

missing = [str(p) for p in required if not p.exists()]
if missing:
    print('THIẾU FILE:')
    for path in missing:
        print('  ', path)
    raise FileNotFoundError('Upload các file còn thiếu lên Drive rồi chạy lại cell này.')
print('Đủ toàn bộ file bắt buộc.')

# --- Bản ucf_train_augment.py trên Drive đã sửa chưa -------------------------------
if '_determinism_requested' not in (SRC_DIR / 'ucf_train_augment.py').read_text(encoding='utf-8'):
    raise RuntimeError(
        'ucf_train_augment.py trên Drive là BẢN CŨ: nó đặt CUBLAS_WORKSPACE_CONFIG vô '
        'điều kiện, khác với mọi lần chạy trước 06/09 kể cả lần cho 88,13. Upload lại '
        f'file đã sửa từ repo lên {SRC_DIR}, rồi chạy lại cell này.')
print('ucf_train_augment.py : bản đã sửa (CUBLAS chỉ đặt khi --deterministic true).')

# --- Dựng cây chạy ------------------------------------------------------------------
if BUILD_DIR.exists():
    shutil.rmtree(BUILD_DIR)
shutil.copytree(SRC_DIR, BUILD_DIR)
for cache in BUILD_DIR.rglob('__pycache__'):
    shutil.rmtree(cache, ignore_errors=True)   # cache cũ theo từ Drive sẽ che file mới
print('Cây chạy   :', BUILD_DIR)

# --- Ghi lại dấu vân tay của đúng thứ sắp chạy --------------------------------------
lines = [f'run_tag : {RUN_TAG}',
         f'python  : {sys.version.split()[0]}',
         'gpu     : ' + subprocess.run(['nvidia-smi', '--query-gpu=name',
                                        '--format=csv,noheader'],
                                       capture_output=True, text=True).stdout.strip()]
try:
    import torch
    lines.append(f'torch   : {torch.__version__}')
except Exception as error:
    lines.append(f'torch   : chưa nạp được ({error})')
for relative in ('ucf_train_augment.py', 'ucf_option_augment.py', 'utils/dataset_augment.py',
                 'model.py', 'utils/tools.py', 'utils/layers.py'):
    digest = hashlib.sha256((BUILD_DIR / relative).read_bytes()).hexdigest()[:16]
    lines.append(f'{relative:<28} sha256:{digest}')
Path(drive_write_path(OUT_DIR / 'environment.txt')).write_text(
    '\n'.join(lines) + '\n', encoding='utf-8')
print()
print('\n'.join(lines))



## 4. Copy Feature Sang Runtime Local — BẮT BUỘC

Đọc hàng nghìn file `.npy` nhỏ trực tiếp từ Drive vừa chậm vừa hay đứt giữa chừng.


In [ ]:

subprocess.run(['df', '-h', '/content'], check=False)
archive = next((p for p in DRIVE_FEATURE_ARCHIVES if p.exists()), None)
started = time.time()
if LOCAL_FEATURE_ROOT.exists() and any(LOCAL_FEATURE_ROOT.iterdir()):
    print('Đã có sẵn ở /content, bỏ qua.')
    FEATURE_ROOT = LOCAL_FEATURE_ROOT
elif archive is not None:
    local_archive = Path('/content') / archive.name
    if not local_archive.exists() or local_archive.stat().st_size != archive.stat().st_size:
        print('Copy archive:', archive)
        shutil.copy2(archive, local_archive)
    print('Giải nén:', local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)
    FEATURE_ROOT = LOCAL_FEATURE_ROOT
elif DRIVE_FEATURE_ROOT.exists():
    print('Không thấy file nén; đọc thẳng từ Drive (chậm hơn nhiều).')
    FEATURE_ROOT = DRIVE_FEATURE_ROOT
else:
    raise FileNotFoundError('Không thấy cả file nén lẫn thư mục feature trên Drive.')

print(f'Xong sau {time.time() - started:.0f}s. FEATURE_ROOT =', FEATURE_ROOT)
print('Số file .npy:', sum(1 for _ in Path(FEATURE_ROOT).rglob('*.npy')))



## 5. Unit Test

Đúng như mục 6 của notebook vòng 1. Khoảng một phút, chạy trên CPU với bộ mã hoá CLIP
giả nên không cần feature thật.


In [ ]:

for test_file in ('test_dataset_augment', 'test_shift_consistency_loss',
                  'test_two_view_batching', 'test_train_smoke'):
    path = BUILD_DIR / 'tests' / f'{test_file}.py'
    if not path.exists():
        print('[không có]', test_file)
        continue
    print('=' * 80)
    result = subprocess.run([sys.executable, '-u', f'tests/{test_file}.py'],
                            cwd=str(BUILD_DIR), capture_output=True, text=True)
    print(result.stdout[-2000:] or result.stderr[-2000:])
    if result.returncode != 0:
        raise RuntimeError(f'{test_file} KHÔNG ĐẠT — dừng lại tìm nguyên nhân.')
print()
print('Tất cả unit test đạt.')



## 6. Chạy — 10 Epoch

Dòng lệnh chép từng cờ một từ `build_train_cmd` của notebook vòng 1.

```
λ = 0 · seed 234 · lr 2e-5 · lô 64 · 10 epoch · MultiStepLR([4,8], 0.1) · train từ đầu
--num-workers 4 · --pin-memory true · --eval-steps 1280
```

Những cờ **không** xuất hiện là cố ý bỏ trống, để script dùng đúng giá trị mặc định mà
vòng 1 đã dùng: `--batch-size 64`, `--shift-ratio 0`, `--shift-direction head`,
`--lambda-auto 0`, `--select-metric auc`, `--deterministic false`,
`--skip-shifted-view false`.

Cũng không truyền `--metrics-csv`: nó làm script chấm thêm một lần mỗi epoch, thành 130
lần chấm thay vì đúng 120 như vòng 1. Lần chấm thêm không đổi trọng số, nhưng khi so đỉnh
với đỉnh thì nhiều lần rút hơn là một lợi thế nhẹ không đáng có. Mục 7 bóc AUC thẳng từ
log, đúng cách con số 88,13 được đọc ra từ log tháng 8.

Log stream trực tiếp ra output của cell và ghi vào `logs/train_ctrl_0909.log`. Theo dõi
dòng `epoch: N | step: … | loss4_raw: …` và `AUC1:` ngay bên dưới nó.


In [ ]:

def build_train_cmd(tag, extra=None):
    """Dòng lệnh của notebook vòng 1, từng cờ một. Chỉ đường dẫn là mới."""
    return [sys.executable, '-u', 'ucf_train_augment.py',
            '--feature-root', str(FEATURE_ROOT),
            '--train-list', TRAIN_LIST,
            '--test-list', TEST_LIST,
            *GT_ARGS,
            '--seed', str(SEED),
            '--lambda-consistency', LAMBDA,
            '--shift-offset', '26',
            '--consistency-branch', 'c',
            '--random-shift', 'false',
            '--consistency-detach', 'false',
            '--consistency-warmup', '1',
            '--max-epoch', str(MAX_EPOCH),
            '--lr', LR,
            '--use-pretrained-model', str(USE_PRETRAINED).lower(),
            '--pretrained-model-path', str(PAPER_MODEL),
            '--num-workers', str(NUM_WORKERS),
            '--pin-memory', 'true',
            '--eval-steps', str(EVAL_STEPS),
            '--output-model-path',    drive_write_path(MODEL_DIR / f'model_{tag}.pth'),
            '--checkpoint-path',      str(SCRATCH / f'checkpoint_{tag}.pth'),
            '--save-cur-path',        str(SCRATCH / f'model_cur_{tag}.pth'),
            '--epoch-checkpoint-dir', str(SCRATCH / f'epoch_checkpoints_{tag}'),
            ] + list(extra or [])


def run_command(cmd, log_name):
    log_path = Path(drive_write_path(LOG_DIR / log_name))
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, cwd=str(BUILD_DIR), stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    log_path.write_text(output, encoding='utf-8')
    print('Log:', log_path)
    if process.returncode != 0:
        raise RuntimeError(f'Lệnh thất bại, mã {process.returncode}')
    return output


target = MODEL_DIR / f'model_{RUN_TAG}.pth'
if target.exists():
    # Chạy lại cell sau khi đã xong thì không train lại từ đầu. Muốn train lại thì
    # xoá file này đi.
    print(f'[bỏ qua] {target} đã tồn tại. Xoá file nếu muốn chạy lại.')
else:
    print('#' * 90)
    print('CHẠY —', RUN_TAG, '| seed', SEED, '| lambda', LAMBDA, '|', MAX_EPOCH, 'epoch')
    print('#' * 90)
    started = time.time()
    run_command(build_train_cmd(RUN_TAG), f'train_{RUN_TAG}.log')
    print()
    print(f'Xong sau {(time.time() - started) / 3600:.2f} giờ')



## 7. Đọc Kết Quả

Bóc đường cong AUC thẳng từ log và so **đỉnh với đỉnh** — vì `--select-metric auc` lưu ra
checkpoint tốt nhất, nên con số ứng với file `.pth` bạn nhận được là đỉnh, không phải lần
chấm cuối.

### Vòng 1 thật sự trông như thế nào

Toàn bộ đường cong của lần chạy 88,13, bóc từ log gốc:

| epoch | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| đỉnh | 86,29 | 87,01 | 87,92 | **87,88** | **87,88** | **88,13** | 88,07 | 88,07 | 88,13 | 88,13 |

Thống kê 96 lần chấm từ epoch 3 trở đi, **trong cùng một lần chạy đó**: trung vị
**87,88**, p75 88,07, cao nhất 88,13.

Tức 88,13 là giá trị lớn nhất trong 120 lần chấm của một đường cong đã bão hoà quanh
87,9. Chính lần chạy đó, ở epoch 4 và 5, đỉnh chỉ đạt 87,88. Đọc con số của bạn cạnh cả
bảng này, đừng đọc cạnh riêng số 88,13.


In [ ]:

import statistics

import pandas as pd

AUC_LINE = re.compile(r'AUC1:\s+([\d.]+)')
ROUND1_EPOCH_PEAKS = [86.29, 87.01, 87.92, 87.88, 87.88, 88.13, 88.07, 88.07, 88.13, 88.13]

log_path = LOG_DIR / f'train_{RUN_TAG}.log'
values = [float(v) * 100
          for v in AUC_LINE.findall(log_path.read_text(encoding='utf-8'))]

per_epoch = max(1, len(values) // MAX_EPOCH)
peaks = [max(values[i * per_epoch:(i + 1) * per_epoch] or [float('nan')])
         for i in range(MAX_EPOCH)]
plateau = values[2 * per_epoch:]        # từ epoch 3 trở đi

table = pd.DataFrame({'epoch': range(1, MAX_EPOCH + 1),
                      'dinh_hom_nay': [round(v, 2) for v in peaks],
                      'dinh_vong_1': ROUND1_EPOCH_PEAKS})
table['chenh'] = (table.dinh_hom_nay - table.dinh_vong_1).round(2)
print(table.to_string(index=False))

print()
print(f'  số lần chấm        : {len(values):>6}      (vòng 1: 120)')
print(f'  ĐỈNH hôm nay       : {max(values):>6.2f}      (vòng 1: 88,13)')
print(f'  điểm cuối hôm nay  : {values[-1]:>6.2f}      (vòng 1: 88,07)')
if len(plateau) > 1:
    print(f'  trung vị từ epoch 3: {statistics.median(plateau):>6.2f}      (vòng 1: 87,88)')

delta = max(values) - 88.13
print()
print(f'  đỉnh hôm nay − 88,13 = {delta:+.2f} điểm')
if abs(delta) <= 0.25:
    print('  => Nằm trong biên độ dao động NỘI BỘ của chính lần chạy vòng 1')
    print('     (đỉnh epoch 4-5 là 87,88, đỉnh epoch 6 là 88,13 — cách nhau 0,25).')
    print('     Không phân biệt được với nhiễu của một lần chạy đơn lẻ.')
else:
    print('  => Vượt biên độ dao động nội bộ của vòng 1. Đáng đào tiếp — nhưng một cặp')
    print('     lần chạy vẫn chưa đủ để khẳng định điều gì về nguyên nhân.')

pd.DataFrame({'eval_index': range(1, len(values) + 1), 'auc': values}).to_csv(
    drive_write_path(OUT_DIR / f'auc_curve_{RUN_TAG}.csv'), index=False)
table.to_csv(drive_write_path(OUT_DIR / f'summary_{RUN_TAG}.csv'), index=False)
print()
print('Saved:', OUT_DIR / f'auc_curve_{RUN_TAG}.csv')
print('Saved:', OUT_DIR / f'summary_{RUN_TAG}.csv')



## Ghi Chú

**Notebook này ghi những gì, ở đâu.** Toàn bộ nằm dưới `Result/shift_loss_09_09/`:

```
environment.txt          run_tag, python, GPU, torch, sha256 của 6 file code chính
summary_ctrl_0909.csv    đỉnh AUC theo epoch, đặt cạnh vòng 1
auc_curve_ctrl_0909.csv  toàn bộ ~120 lần chấm
logs/train_ctrl_0909.log nhật ký đầy đủ, có quỹ đạo loss4_raw
models/model_ctrl_0909.pth
```

`drive_write_path()` chặn mọi đường ghi nằm ngoài thư mục đó. Checkpoint từng epoch và
file trung gian đi vào `/content/scratch`, mất khi phiên kết thúc — mỗi cái 350 MB và
không cần cho việc này. Cần phân tích theo lớp (`ucf_analyze_checkpoints.py`) thì đổi
`--epoch-checkpoint-dir` sang `MODEL_DIR` và canh dung lượng Drive.

**Không có resume.** Phiên Colab đứt giữa chừng là mất lần chạy.
`--use-pretrained-model` nạp trọng số nhưng vòng lặp vẫn chạy từ epoch 0 với optimizer và
scheduler mới, nên nó không phải resume.

**`--num-workers` phải giữ nguyên 4.** Không phải vì tốc độ: khi `num_workers > 0`,
DataLoader rút thêm một số ngẫu nhiên toàn cục để gieo hạt cho worker, nên thứ tự lô thay
đổi theo. Đổi số này là đổi luôn dữ liệu mà mô hình nhìn thấy, và lần chạy hết so được
với vòng 1.

**Giới hạn.** Đây là **một** lần chạy, **một** hạt giống, trên **một** GPU. Vòng 2 đo
được 2,56 điểm AUC giữa hai lần đối chứng chỉ khác hạt giống — lớn hơn mọi hiệu ứng đang
xét. Con số ở đây trả lời được "code hiện tại chạy ra khoảng bao nhiêu", không trả lời
được "vì sao lệch".
